# M2 远程 PPO 云 Worker — Colab 冒烟测试

连接到 hub 的 cloudflared tunnel，拉取 job 执行 PPO 更新。

## 使用前

1. 确保 hub 端已启动：`hub-server` + `cloudflared tunnel` + `training loop`
2. 在下方的 `⚙️ 参数配置` 单元格填入当前 tunnel URL 和 token
3. 依次运行各单元格

---
## ⚙️ 参数配置

In [ ]:
# @title 填入 hub 连接信息
HUB_URL = "https://kent-decorating-perceived-cope.trycloudflare.com"  # @param {type:"string"}
HUB_TOKEN = "m2-test-token-74924333"  # @param {type:"string"}
COMMIT = "824ace254566748f01d724a8b56ff2818e247663"  # @param {type:"string"}
BRANCH = "goal-nn"  # @param {type:"string"}
# Git 仓库 URL（私库需嵌入 PAT：https://<user>:<pat>@github.com/huangjian/battle2.git）
GIT_REPO = "https://github.com/huangjian/battle2.git"  # @param {type:"string"}

# 保活配置
KEEPALIVE_HOURS = 2  # @param {type:"integer"}
POLL_INTERVAL_SEC = 5  # @param {type:"integer"}

print(f"HUB_URL = {HUB_URL}")
print(f"COMMIT = {COMMIT}")
print(f"Keepalive = {KEEPALIVE_HOURS}h")

---
## 1. 克隆仓库（shallow，pin commit）

> 仓库私有，需在 ⚙️ 参数配置中设置 `GIT_REPO`（含 PAT）：`https://<user>:<pat>@github.com/huangjian/battle2.git`

In [ ]:
import subprocess
import sys
import time
from pathlib import Path

REPO_DIR = "/content/battle2"

if not Path(REPO_DIR).exists():
    print(f"Cloning {GIT_REPO} (shallow, depth 1, branch={BRANCH})...")
    r = subprocess.run([
        "git", "clone", "--depth", "1",
        "--branch", BRANCH,
        GIT_REPO,
        REPO_DIR
    ], capture_output=True, text=True)
    if r.returncode != 0:
        print(f"git clone FAILED (rc={r.returncode})")
        print(f"stdout: {r.stdout[-500:]}")
        print(f"stderr: {r.stderr[-500:]}")
        raise SystemExit(r.returncode)
    r2 = subprocess.run(
        ["git", "checkout", COMMIT],
        cwd=REPO_DIR, capture_output=True, text=True
    )
    if r2.returncode != 0:
        print(f"commit {COMMIT[:12]} not in shallow clone, fetching...")
        subprocess.run(["git", "fetch", "--depth", "100"], cwd=REPO_DIR, check=True)
        subprocess.run(["git", "checkout", COMMIT], cwd=REPO_DIR, check=True)
    print(f"Checked out {COMMIT[:12]}")
else:
    print(f"Repo already exists at {REPO_DIR}")

%cd {REPO_DIR}/nn-training

---
## 2. 安装依赖

In [ ]:
import subprocess

import torch


def run(cmd, **kw):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True, **kw)

run(f"{sys.executable} -m pip install --quiet torch numpy")

print(f"torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  device: {torch.cuda.get_device_name(0)}")

print("Dependencies ready")

---
## 3. 保活线程（防止 Colab 90min 超时）

Colab 免费版约 90min 回收空闲会话。通过定时 ping 界面保持活跃。

In [ ]:
import threading

from IPython.display import Javascript
from IPython.display import display as ipy_display

KEEPALIVE_STOP = threading.Event()

def keepalive_loop():
    """每 60s 点一次 Colab 的 connect 防止超时。"""
    n = 0
    while not KEEPALIVE_STOP.is_set():
        try:
            ipy_display(Javascript("""
                function clickConnect() {
                    document.querySelector("colab-connect-button")?.click();
                }
                setTimeout(clickConnect, 1000);
            """))
            n += 1
        except Exception:
            pass
        KEEPALIVE_STOP.wait(60)
    print(f"Keepalive stopped after {n} pings")

th = threading.Thread(target=keepalive_loop, daemon=True, name="keepalive")
th.start()
print(f"Keepalive thread started (every 60s, max {KEEPALIVE_HOURS}h)")

---
## 4. 启动远程 PPO Worker

轮询 hub 获取 job → 下载 payload → 在 GPU 上跑 PPO → POST 结果回 hub。

首次启动会 import torch（~5-10s）。之后每处理一个 job 会回到轮询状态。

> 可以随时中断此单元格（`Runtime → Interrupt execution`），worker 会优雅退出。

In [ ]:
from pathlib import Path

WORK_DIR = Path("/content/remote-worker")
WORK_DIR.mkdir(parents=True, exist_ok=True)

nn_root = Path.cwd()
if str(nn_root) not in sys.path:
    sys.path.insert(0, str(nn_root))

from remote.worker import worker_loop

print(f"[colab] Connecting to hub: {HUB_URL}")
print(f"[colab] Work dir: {WORK_DIR}")
print(f"[colab] Poll interval: {POLL_INTERVAL_SEC}s")
print(f"[colab] Keepalive: {KEEPALIVE_HOURS}h")
print("[colab] Starting worker loop...")

t_start = time.time()

try:
    n = worker_loop(
        HUB_URL,
        HUB_TOKEN,
        work_dir=WORK_DIR,
        device="cuda",
        torch_threads=0,
        poll_sec=POLL_INTERVAL_SEC,
        once=False,
        max_idle_sec=KEEPALIVE_HOURS * 3600,
    )
    print(f"\n[colab] Worker exited: {n} job(s) processed")
except KeyboardInterrupt:
    print("\n[colab] Worker interrupted by user")
finally:
    KEEPALIVE_STOP.set()

elapsed = time.time() - t_start
print(f"[colab] Session duration: {elapsed/60:.1f} min")

---
## 5. 停止保活

如果提前中断了 worker，运行此单元格停止保活线程。

In [ ]:
KEEPALIVE_STOP.set()
print("Keepalive stopped")

---
## 附录：预期日志

hub 端：
```
[hub-server] "GET /jobs/next HTTP/1.1" 200 -
[hub-server] "GET /jobs/{id}/payload HTTP/1.1" 200 -
[hub-server] "POST /jobs/{id}/result HTTP/1.1" 200 -
```

worker 端：
```
[worker] job {id} claimed — downloading payload
[worker] job {id}: PPO done in {sec}s, steps={n} chunks={m} kl={k}
[worker] job {id} done — result accepted
```